# Hands-On 5: Build Custom CrewAI Tools

### Definition
A **CrewAI Tool** gives an agent capabilities beyond normal text generation, such as calculations, file access, or external data lookup.

### Goal
Create tools, assign them to agents, execute tool calls, process results, and handle errors.

In [2]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))

OpenRouter key loaded: True
Model: openai/gpt-4o-mini


In [3]:
llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
print("LLM:", llm)

LLM: openai/gpt-4o-mini


## 1. Calculator Tool

Calculates an expression and returns a readable result or an error.

In [10]:
from crewai.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression safely."""
    try:
        allowed_names = {
            "abs": abs,
            "round": round,
            "max": max,
            "min": min,
            "sum": sum,
            "pow": pow,
            "len": len
        }

        result = eval(
            expression,
            {"__builtins__": {}},
            allowed_names
        )

        return f"Result: {result}"

    except Exception as e:
        return f"Error: {e}. Please provide a valid expression."


# Test CrewAI tool
print(
    calculator.run("120 * 8 * 45")
)

Result: 43200


## 2. File-Reading Tool

Reads a text file and handles missing-file errors.

In [11]:
@tool
def read_file_content(file_path: str) -> str:
    """Read and return a text file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"Error: file not found: {file_path}"
    except Exception as e:
        return f"Error reading file: {e}"

## 3. API Tool

This uses a small mock weather API, so no extra API key is required.

In [12]:
@tool
def fetch_weather_data(city: str) -> str:
    """Fetch weather data for a city from a mock API."""
    weather_db = {
        "london": {"temp": 15, "condition": "Rainy", "humidity": 80},
        "tokyo": {"temp": 20, "condition": "Cloudy", "humidity": 60},
        "new york": {"temp": 22, "condition": "Sunny", "humidity": 45},
        "paris": {"temp": 18, "condition": "Partly Cloudy", "humidity": 55},
    }

    city = city.lower().strip()

    if city not in weather_db:
        return f"Error: weather unavailable for {city}."

    data = weather_db[city]
    return (
        f"Weather for {city.title()}: "
        f"{data['temp']}°C, {data['condition']}, "
        f"humidity {data['humidity']}%"
    )

## 4. Company Lookup Tool

A small mock data lookup tool based on the reference notebook.

In [13]:
@tool
def search_company_info(company_name: str) -> str:
    """Look up company information from a mock database."""
    companies = {
        "openai": {"founded": 2015, "ceo": "Sam Altman", "headquarters": "San Francisco, CA"},
        "google": {"founded": 1998, "ceo": "Sundar Pichai", "headquarters": "Mountain View, CA"},
        "microsoft": {"founded": 1975, "ceo": "Satya Nadella", "headquarters": "Redmond, WA"}
    }

    name = company_name.lower().strip()

    if name not in companies:
        return f"Error: company not found: {company_name}"

    data = companies[name]
    return (
        f"Company: {company_name.title()} | "
        f"Founded: {data['founded']} | "
        f"CEO: {data['ceo']} | "
        f"Headquarters: {data['headquarters']}"
    )

## 5. Create Sample File

In [15]:
sample_content = """Project Update - Q1 2025
Team: AI Research Division
Status: On Track
Key Achievements:
1. Launched new LLM evaluation framework
2. Reduced inference latency by 40%
3. Published 3 research papers
Next Steps:
- Deploy multi-agent system to production
- Expand training dataset by 2x
"""

with open("sample_project.txt", "w", encoding="utf-8") as f:
    f.write(sample_content)

print(read_file_content.run("sample_project.txt"))

Project Update - Q1 2025
Team: AI Research Division
Status: On Track
Key Achievements:
1. Launched new LLM evaluation framework
2. Reduced inference latency by 40%
3. Published 3 research papers
Next Steps:
- Deploy multi-agent system to production
- Expand training dataset by 2x



## 6. Assign Tools to Agents

In [16]:
math_agent = Agent(
    role="Mathematical Analyst",
    goal="Perform accurate calculations using the calculator tool.",
    backstory="A precise analyst who verifies calculations instead of guessing.",
    llm=llm,
    tools=[calculator],
    verbose=True,
    allow_delegation=False
)

data_agent = Agent(
    role="Data Researcher",
    goal="Gather information from files and data-access tools.",
    backstory="A researcher skilled at extracting and combining information from different sources.",
    llm=llm,
    tools=[read_file_content, fetch_weather_data, search_company_info],
    verbose=True,
    allow_delegation=False
)

print("Math tools:", [t.name for t in math_agent.tools])
print("Data tools:", [t.name for t in data_agent.tools])

Math tools: ['calculator']
Data tools: ['read_file_content', 'fetch_weather_data', 'search_company_info']


## 7. Test Individual Tools

In [19]:
print("Calculator:", calculator.run("(120 * 8) * 45"))

print("File:", read_file_content.run("sample_project.txt"))

print("Weather:", fetch_weather_data.run("London"))

print("Company:", search_company_info.run("OpenAI"))

Calculator: Result: 43200
File: Project Update - Q1 2025
Team: AI Research Division
Status: On Track
Key Achievements:
1. Launched new LLM evaluation framework
2. Reduced inference latency by 40%
3. Published 3 research papers
Next Steps:
- Deploy multi-agent system to production
- Expand training dataset by 2x

Weather: Weather for London: 15°C, Rainy, humidity 80%
Company: Company: Openai | Founded: 2015 | CEO: Sam Altman | Headquarters: San Francisco, CA


## 8. Execute Tool-Enabled Crew

In [20]:
math_task = Task(
    description=(
        "Calculate the daily payroll for 120 employees working 8 hours "
        "per day at $45/hour, then calculate the monthly cost for 22 days."
    ),
    expected_output="Daily and monthly payroll costs.",
    agent=math_agent
)

data_task = Task(
    description=(
        "Read sample_project.txt and summarize it. "
        "Get weather for London and Tokyo. "
        "Look up OpenAI company information. "
        "Combine the findings into one concise report."
    ),
    expected_output="A concise report containing all requested findings.",
    agent=data_agent
)

tool_crew = Crew(
    agents=[math_agent, data_agent],
    tasks=[math_task, data_task],
    process=Process.sequential,
    verbose=True,
    memory=False
)

print("Tool-enabled crew created")

Tool-enabled crew created


In [21]:
print("=" * 60)
print("EXECUTING TOOL-ENABLED CREW")
print("=" * 60)

result = await tool_crew.kickoff_async()

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

EXECUTING TOOL-ENABLED CREW


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cf6f34b2-b5d4-48b0-9446-d5493b8a12fc                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Calculate the daily payroll for 120 employees working 8 hours per day at $45/hour, then calculate the    │
│  monthly cost for 22 days.                                                                                      │
│  ID: 211b804c-91d0-418c-9888-e16c95596862                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Mathematical Analyst                                                                                    │
│                                                                                                                 │
│  Task: Calculate the daily payroll for 120 employees working 8 hours per day at $45/hour, then calculate the    │
│  monthly cost for 22 days.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: calculator                                                                                               │
│  Args: {'expression': '120 * 8 * 45'}                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool calculator executed with result: Result: 43200...
Tool calculator executed with result: Result: 950400...


... (19 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Read sample_project.txt and summarize it. Get weather for London and Tokyo. Look up OpenAI company       │
│  information. Combine the findings into one concise report.                                                     │
│  Agent: Data Researcher                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cf6f34b2-b5d4-48b0-9446-d5493b8a12fc                                                                       │
│  Final Output: **Concise Report**                                                                               │
│                                                                                                                 │
│  **1. Project Summary:**                                                                                        │
│  - **Title:** Project Update - Q1 2025                                                                          │
│  - **Team:** AI Research Division                                                                               │
│  - **Status:** On Track                                                                                         │
│  - **Key Achievements:**                                                                                        │
│    1. Launched new LLM evaluation framework                                                                     │
│    2. Reduced inference latency by 40%                                                                          │
│    3. Published 3 research papers                                                                               │
│  - **Next Steps:**                                                                                              │
│    - Deploy multi-agent system to production                                                                    │
│    - Expand training dataset by 2x                                                                              │
│                                                                                                                 │
│  **2. Weather Information:**                                                                                    │
│  - **London:** 15°C, Rainy, humidity 80%                                                                        │
│  - **Tokyo:** 20°C, Cloudy, humidity 60%                                                                        │
│                                                                                                                 │
│  **3. Company Information:**                                                                                    │
│  - **Company Name:** OpenAI                                                                                     │
│  - **Founded:** 2015                                                                                            │
│  - **CEO:** Sam Altman                                                                                          │
│  - **Headquarters:** San Francisco, CA                                                                          │
│                                                                                                                 │
│  **Contextual Financial Information:**                                                                          │
│  - The daily payroll cost for 120 employees working 8 hours per day at $45/hour is **$43,200**.                 │
│  - The monthly cost for 22 days is **$950,400**.                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────


FINAL OUTPUT
**Concise Report**

**1. Project Summary:**
- **Title:** Project Update - Q1 2025
- **Team:** AI Research Division
- **Status:** On Track
- **Key Achievements:**
  1. Launched new LLM evaluation framework
  2. Reduced inference latency by 40%
  3. Published 3 research papers
- **Next Steps:**
  - Deploy multi-agent system to production
  - Expand training dataset by 2x

**2. Weather Information:**
- **London:** 15°C, Rainy, humidity 80%
- **Tokyo:** 20°C, Cloudy, humidity 60%

**3. Company Information:**
- **Company Name:** OpenAI
- **Founded:** 2015
- **CEO:** Sam Altman
- **Headquarters:** San Francisco, CA

**Contextual Financial Information:**
- The daily payroll cost for 120 employees working 8 hours per day at $45/hour is **$43,200**.
- The monthly cost for 22 days is **$950,400**.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 9. Tool Error Handling

Tools return controlled error messages instead of crashing the workflow.

In [23]:
print(calculator.run("10 / 0"))
print(read_file_content.run("missing_file.txt"))
print(fetch_weather_data.run("Bengaluru"))
print(search_company_info.run("UnknownCompany"))

Error: division by zero. Please provide a valid expression.
Error: file not found: missing_file.txt
Error: weather unavailable for bengaluru.
Error: company not found: UnknownCompany


## 10. Key Points

- `@tool` converts a Python function into an agent tool.
- Tools should have clear descriptions and typed parameters.
- Assign tools with `tools=[...]`.
- Agents can call tools during task execution.
- Handle tool failures with useful error messages.
- Keep each tool focused on one purpose.

### Completed
- ✅ Calculator tool
- ✅ File-reading tool
- ✅ API-style tool
- ✅ Company lookup tool
- ✅ Tools assigned to agents
- ✅ Tool calls executed
- ✅ Tool output processed
- ✅ Tool errors handled